# Explorar dados raspados
Base atual (`eventos.db`, SQLite). Rode as células e mexa à vontade.

In [ ]:
import sqlite3, pandas as pd
from pathlib import Path
_p = Path.cwd()
DB = next(d/"eventos.db" for d in [_p, *_p.parents] if (d/"eventos.db").exists())
con = sqlite3.connect(DB)
q = lambda sql, *a: pd.read_sql_query(sql, con, params=a)
pd.set_option("display.max_colwidth", 70)
q("SELECT fonte, COUNT(*) n FROM eventos GROUP BY fonte ORDER BY n DESC")

In [ ]:
# todos os eventos, por data
q("SELECT fonte, start_date, nome, local_nome, cidade, url FROM eventos ORDER BY start_date")

In [ ]:
# busca textual (mesmo índice FTS que o agente usaria)
def buscar(termo):
    return q("SELECT fonte, start_date, nome, local_nome, url FROM eventos "
             "WHERE rowid IN (SELECT rowid FROM eventos_fts WHERE eventos_fts MATCH ?) "
             "ORDER BY start_date", termo)
buscar("pagode")

In [ ]:
# próximos eventos a partir de agora
from datetime import datetime, timezone
agora = datetime.now(timezone.utc).isoformat()
q("SELECT fonte, start_date, nome, local_nome FROM eventos WHERE start_date>=? ORDER BY start_date LIMIT 30", agora)

In [ ]:
# célula livre — escreva seu próprio SQL, ex.:
q("SELECT * FROM eventos WHERE nome LIKE '%funk%'")